# TCGer iOS Scan Index — Parity Rebuild V2 (Colab)

Rebuilds `CardsIndexVectors.bin` for the iOS scanner by re-embedding every catalog card image with the **same encoder the phone runs**: `facebook/dinov2-small` in fp32 (verified ≈0.999 cosine vs the CoreML fp16 conversion), using the exact iOS preprocessing (resize shortest edge 256 → center crop 224 → ImageNet norm). The shipped index was built with a q8-quantized ONNX model, which costs ~0.02–0.09 of similarity per card; this rebuild removes that gap.

**Inputs** (place in the Drive folder or upload to `/content/`):
1. Card images — already in Drive (`card-library/pokemon/...`, any layout; matched by filename).
2. `CardsIndexMetadata.json` — from the repo at `mobile-apps/ios/TCGer/TCGer/Resources/ScanIndex/` (defines row order).
3. `CardsIndexVectors.bin` (optional but recommended) — the current bin; rows whose image can't be found/fetched keep their old vector.

**Output**: `CardsIndexVectors-parity-v2.bin` written atomically to the Drive folder only after validation passes. Copy it into the repo as `ScanIndex/CardsIndexVectors.bin` and rebuild the app. The **web** index must stay q8-built (the browser runtime is q8) — this bin is iOS-only.

Progress is checkpointed to Drive every 2,000 attempted rows, so a disconnected runtime resumes with at most one checkpoint interval to repeat. V2 rejects checkpoints from different metadata, model, preprocessing, or library versions and retries failed rows. Use a GPU runtime (Runtime → Change runtime type → T4).

In [ ]:
# ---- Config ----
DRIVE_FOLDER = "/content/drive/MyDrive/UniFi Drive_UNAS Pro 8/UNAS Pro 8_Main Backup/Images/pvc-19daba96-3902-4005-aab6-60b80b8f171a/card-library"
IMAGES_SUBDIR = "pokemon"                               # subfolder holding the images
MODEL_ID = "facebook/dinov2-small"
MODEL_REVISION = "ed25f3a31f01632728cabb09d1542f84ab7b0056"

# Metadata / old bin: first path that exists wins.
METADATA_CANDIDATES = [DRIVE_FOLDER + "/pokemon/CardsIndexMetadata.json", DRIVE_FOLDER + "/CardsIndexMetadata.json", "/content/CardsIndexMetadata.json"]
OLD_BIN_CANDIDATES  = [DRIVE_FOLDER + "/pokemon/CardsIndexVectors.bin", DRIVE_FOLDER + "/CardsIndexVectors.bin", "/content/CardsIndexVectors.bin"]

OUT_BIN = DRIVE_FOLDER + "/CardsIndexVectors-parity-v2.bin"
CHECKPOINT = DRIVE_FOLDER + "/parity-rebuild-v2-checkpoint.npz"

COPY_IMAGES_LOCAL = False   # direct Drive reads suit one pass; True repairs/resumes a local rsync
DOWNLOAD_MISSING = True     # fetch images missing from Drive from their imageURL
BATCH_SIZE = 64
LOADER_THREADS = 16
LOAD_WINDOW = 256           # bound decoded-image memory while keeping loader threads busy
CHECKPOINT_EVERY = 2000
CHECKPOINT_SCHEMA = 2
SCALE = 127                 # int8 quantization scale — must match AnnoyIndexStore

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os, json, struct, math, hashlib
import numpy as np

meta_path = next((p for p in METADATA_CANDIDATES if os.path.exists(p)), None)
assert meta_path, f"CardsIndexMetadata.json not found in {METADATA_CANDIDATES} — copy it from the repo's ScanIndex folder"
with open(meta_path, "rb") as f:
    meta_bytes = f.read()
meta = json.loads(meta_bytes)
assert all(e["annIndex"] == i for i, e in enumerate(meta)), "annIndex order mismatch"
N = len(meta)
metadata_sha256 = hashlib.sha256(meta_bytes).hexdigest()
print(f"metadata: {N} cards from {meta_path} (sha256 {metadata_sha256[:12]}...)")

old_q = None
old_path = next((p for p in OLD_BIN_CANDIDATES if os.path.exists(p)), None)
if old_path:
    with open(old_path, "rb") as f:
        raw = f.read()
    assert len(raw) >= 8, f"old bin is truncated: {old_path}"
    cnt, dim = struct.unpack("<ii", raw[:8])
    assert cnt == N, f"old bin has {cnt} rows, metadata has {N}"
    assert dim == 384, f"old bin dimension is {dim}, expected 384"
    assert len(raw) == 8 + cnt * dim, f"old bin size mismatch: expected {8 + cnt * dim}, got {len(raw)}"
    old_q = np.frombuffer(raw, dtype=np.int8, offset=8, count=cnt*dim).reshape(cnt, dim).copy()
    print(f"old bin: {cnt} x {dim} from {old_path} (fallback rows available)")
else:
    dim = 384
    print("no old bin found — every image must embed successfully before V2 will publish output")

In [ ]:
# ---- Locate images: recursive scan, match files to cardIds by common layouts ----
import pathlib, subprocess

src_root = os.path.join(DRIVE_FOLDER, IMAGES_SUBDIR)
assert os.path.isdir(src_root), f"images folder not found: {src_root}"

root = src_root
if COPY_IMAGES_LOCAL:
    root = "/content/card-images-v2"
    os.makedirs(root, exist_ok=True)
    print("syncing images to local disk (safe to re-run after an interrupted copy)...")
    subprocess.run(["rsync", "-a", "--partial", "--info=progress2", src_root + "/", root + "/"], check=True)
    print("using local copy:", root)

EXTS = {".png", ".jpg", ".jpeg", ".webp"}
files = [p for p in pathlib.Path(root).rglob("*") if p.suffix.lower() in EXTS]
print(f"found {len(files)} image files")

# Index files by candidate keys:
#   1. bare stem            e.g. sv03-136.webp            -> "sv03-136"
#   2. parent/stem          e.g. sv03/136.png             -> "sv03-136"
#   3. stem 'high'/'low' with card folder: sv03/136/high.webp -> "sv03-136"
by_key = {}
for p in files:
    stem = p.stem
    by_key.setdefault(stem, p)
    by_key.setdefault(f"{p.parent.name}-{stem}", p)
    if stem in ("high", "low"):
        by_key.setdefault(f"{p.parent.parent.name}-{p.parent.name}", p)

def find_image(card_id):
    for key in (card_id, card_id.replace("/", "_")):
        if key in by_key:
            return by_key[key]
    return None

have = sum(1 for e in meta if find_image(e["cardId"]) is not None)
print(f"matched {have}/{N} cards to Drive images ({N-have} missing{' — will download' if DOWNLOAD_MISSING else ''})")
missing_sample = [e["cardId"] for e in meta if find_image(e["cardId"]) is None][:10]
print("sample missing:", missing_sample)

In [ ]:
# ---- Model: pinned DINOv2-small fp32, CLS token, iOS-identical preprocessing ----
import torch
import transformers
from transformers import Dinov2Model
from PIL import Image, __version__ as PILLOW_VERSION

device = "cuda" if torch.cuda.is_available() else "cpu"
assert device == "cuda", "V2 requires a Colab GPU runtime (Runtime → Change runtime type → T4)"
model = Dinov2Model.from_pretrained(MODEL_ID, revision=MODEL_REVISION).eval().to(device)
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1).to(device)
STD  = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1).to(device)
print(f"device: {device}; model: {MODEL_ID}@{MODEL_REVISION[:12]}...")

def preprocess(img):
    """Mirror CardEmbeddingEncoder.swift: resize shortest edge 256 (bicubic,
    ceil), center crop 224. Returns HWC uint8 ndarray."""
    w, h = img.size
    s = max(256 / min(w, h), 224 / w, 224 / h)
    rw, rh = math.ceil(w * s), math.ceil(h * s)
    img = img.resize((rw, rh), Image.BICUBIC)
    cx, cy = max(0, (rw - 224) // 2), max(0, (rh - 224) // 2)
    return np.asarray(img.crop((cx, cy, cx + 224, cy + 224)))

@torch.no_grad()
def embed_batch(arrs):
    x = torch.from_numpy(np.stack(arrs)).to(device).permute(0, 3, 1, 2).float() / 255.0
    x = (x - MEAN) / STD
    cls = model(x).last_hidden_state[:, 0]
    return torch.nn.functional.normalize(cls, dim=-1).cpu().numpy()

In [ ]:
# ---- Embed all rows (checkpointed, resumable, provenance-checked) ----
import time, urllib.request, concurrent.futures

assert BATCH_SIZE > 0 and LOADER_THREADS > 0 and LOAD_WINDOW >= BATCH_SIZE
run_manifest = {
    "checkpointSchema": CHECKPOINT_SCHEMA,
    "metadataSHA256": metadata_sha256,
    "rowCount": N,
    "dimension": dim,
    "modelId": MODEL_ID,
    "modelRevision": MODEL_REVISION,
    "preprocessing": "shortest-edge-256_ceil-center-crop-224_imagenet-norm",
    "scale": SCALE,
    "imagesRoot": f"{DRIVE_FOLDER}/{IMAGES_SUBDIR}",
    "numpyVersion": np.__version__,
    "torchVersion": torch.__version__,
    "transformersVersion": transformers.__version__,
    "pillowVersion": PILLOW_VERSION,
}
manifest_json = json.dumps(run_manifest, sort_keys=True, separators=(",", ":"))
run_fingerprint = hashlib.sha256(manifest_json.encode()).hexdigest()
print(f"run fingerprint: {run_fingerprint[:12]}...")

new_q = old_q.copy() if old_q is not None else np.zeros((N, dim), dtype=np.int8)
embedded = np.zeros(N, dtype=bool)
failed = np.zeros(N, dtype=bool)

def atomic_save_checkpoint():
    tmp = CHECKPOINT + ".tmp.npz"
    np.savez(
        tmp, new_q=new_q, embedded=embedded, failed=failed,
        fingerprint=np.array(run_fingerprint), manifest=np.array(manifest_json),
    )
    os.replace(tmp, CHECKPOINT)

if os.path.exists(CHECKPOINT):
    try:
        with np.load(CHECKPOINT, allow_pickle=False) as ck:
            required = {"new_q", "embedded", "failed", "fingerprint", "manifest"}
            missing = required.difference(ck.files)
            if missing:
                raise ValueError(f"missing fields: {sorted(missing)}")
            stored_fingerprint = str(ck["fingerprint"].item())
            if stored_fingerprint != run_fingerprint:
                stored_manifest = str(ck["manifest"].item())
                raise ValueError(
                    f"provenance mismatch\nexpected: {manifest_json}\nfound:    {stored_manifest}"
                )
            loaded_q = ck["new_q"].copy()
            loaded_embedded = ck["embedded"].copy()
            loaded_failed = ck["failed"].copy()
        if loaded_q.shape != (N, dim) or loaded_q.dtype != np.int8:
            raise ValueError(f"invalid new_q shape/dtype: {loaded_q.shape} {loaded_q.dtype}")
        if loaded_embedded.shape != (N,) or loaded_embedded.dtype != np.bool_:
            raise ValueError(f"invalid embedded shape/dtype: {loaded_embedded.shape} {loaded_embedded.dtype}")
        if loaded_failed.shape != (N,) or loaded_failed.dtype != np.bool_:
            raise ValueError(f"invalid failed shape/dtype: {loaded_failed.shape} {loaded_failed.dtype}")
        if np.any(loaded_embedded & loaded_failed):
            raise ValueError("rows cannot be both embedded and failed")
        new_q, embedded, failed = loaded_q, loaded_embedded, loaded_failed
        print(f"resuming: {int(embedded.sum())}/{N} embedded, {int(failed.sum())} failed rows will retry")
    except Exception as exc:
        raise RuntimeError(
            f"Checkpoint is incompatible or corrupt: {CHECKPOINT}\n"
            "Move/delete it or choose a new CHECKPOINT path before continuing.\n"
            f"Details: {exc}"
        ) from exc

dl_dir = "/content/downloaded-images-v2"
os.makedirs(dl_dir, exist_ok=True)

def load_row(i):
    entry = meta[i]
    path = find_image(entry["cardId"])
    downloaded = False
    if path is None and DOWNLOAD_MISSING and entry.get("imageURL"):
        key = hashlib.sha256(f"{i}:{entry['cardId']}".encode()).hexdigest()[:20]
        local = os.path.join(dl_dir, key + ".img")
        tmp = local + ".part"
        try:
            if not os.path.exists(local):
                request = urllib.request.Request(
                    entry["imageURL"], headers={"User-Agent": "TCGer-index-rebuild-v2"}
                )
                with urllib.request.urlopen(request, timeout=30) as response, open(tmp, "wb") as f:
                    f.write(response.read())
                os.replace(tmp, local)
            path = local
            downloaded = True
        except Exception:
            if os.path.exists(tmp):
                os.remove(tmp)
            return i, None
    if path is None:
        return i, None
    try:
        with Image.open(path) as image:
            return i, preprocess(image.convert("RGB"))
    except Exception:
        # A cached partial/corrupt download should be fetched again next run.
        if downloaded and os.path.exists(path):
            os.remove(path)
        return i, None

# Only successful embeddings are terminal. Failed rows retry on every resume.
pending = np.flatnonzero(~embedded).tolist()
failed[pending] = False
print(f"embedding {len(pending)} rows...")
t0, processed, last_ckpt = time.time(), 0, 0
batch_idx, batch_arr = [], []

def flush_batch():
    global processed
    if not batch_idx:
        return
    vectors = embed_batch(batch_arr)
    for row_index, vector in zip(batch_idx, vectors):
        new_q[row_index] = np.clip(np.round(vector * SCALE), -127, 127).astype(np.int8)
        embedded[row_index] = True
        failed[row_index] = False
    processed += len(batch_idx)
    batch_idx.clear()
    batch_arr.clear()

with concurrent.futures.ThreadPoolExecutor(max_workers=LOADER_THREADS) as pool:
    for window_start in range(0, len(pending), LOAD_WINDOW):
        window = pending[window_start:window_start + LOAD_WINDOW]
        for row_index, array in pool.map(load_row, window):
            if array is None:
                failed[row_index] = True
                processed += 1
            else:
                batch_idx.append(row_index)
                batch_arr.append(array)
                if len(batch_idx) >= BATCH_SIZE:
                    flush_batch()
            if processed - last_ckpt >= CHECKPOINT_EVERY:
                last_ckpt = processed
                atomic_save_checkpoint()
                rate = processed / max(time.time() - t0, 1e-9)
                eta = (len(pending) - processed) / max(rate, 1) / 60
                print(
                    f"{int(embedded.sum())}/{N} embedded (failed {int(failed.sum())}) — "
                    f"{rate:.0f}/s, eta {eta:.0f} min"
                )
    flush_batch()
atomic_save_checkpoint()
print(f"done: {int(embedded.sum())}/{N} embedded, {int(failed.sum())} fallback rows")
if not embedded.any():
    raise RuntimeError("every row failed — check image paths/network; output was not written")
if failed.any() and old_q is None:
    raise RuntimeError("some rows failed and no old bin is available for fallback; output was not written")

In [ ]:
# ---- Verify first, then atomically publish the bin ----
if new_q.shape != (N, dim) or new_q.dtype != np.int8:
    raise RuntimeError(f"invalid output matrix: {new_q.shape} {new_q.dtype}")
if not embedded.any():
    raise RuntimeError("0 rows were embedded; output was not written")
if failed.any() and old_q is None:
    raise RuntimeError("failed rows have no fallback vectors; output was not written")

a = new_q.astype(np.float32) / SCALE
norms = np.linalg.norm(a, axis=1, keepdims=True)
zero_rows = np.flatnonzero(norms[:, 0] <= 1e-12)
if len(zero_rows):
    raise RuntimeError(f"{len(zero_rows)} zero-vector rows found (sample {zero_rows[:10].tolist()}); output was not written")
a /= norms

if old_q is not None:
    successful_rows = np.flatnonzero(embedded)
    changed_count = int(np.any(new_q[successful_rows] != old_q[successful_rows], axis=1).sum())
    if changed_count == 0:
        raise RuntimeError("all embedded rows equal the old index; output was not written")
    b = old_q.astype(np.float32) / SCALE
    b /= np.maximum(np.linalg.norm(b, axis=1, keepdims=True), 1e-12)
    cosine = (a[successful_rows] * b[successful_rows]).sum(axis=1)
    print(
        f"new-vs-old agreement on {len(successful_rows)} embedded rows "
        f"({changed_count} changed after q8): mean {cosine.mean():.3f}, "
        f"p5 {np.percentile(cosine, 5):.3f} (expected ~0.95 mean)"
    )

# Sampled self-retrieval catches row-order and gross embedding corruption.
successful_rows = np.flatnonzero(embedded)
sample = successful_rows[::max(1, len(successful_rows) // 500)]
top1 = (a[sample] @ a.T).argmax(axis=1)
self_retrieval = float((top1 == sample).mean())
print(f"self-retrieval on {len(sample)} sampled rows: {self_retrieval*100:.1f}% top-1")
if self_retrieval < 0.99:
    raise RuntimeError("self-retrieval below 99%; output was not written")

tmp_out = OUT_BIN + ".tmp"
with open(tmp_out, "wb") as f:
    f.write(struct.pack("<ii", N, dim))
    f.write(new_q.tobytes())
expected_size = 8 + N * dim
if os.path.getsize(tmp_out) != expected_size:
    raise RuntimeError("temporary output size mismatch; existing output was not replaced")
os.replace(tmp_out, OUT_BIN)
with open(OUT_BIN, "rb") as f:
    output_sha256 = hashlib.sha256(f.read()).hexdigest()
print(f"published {OUT_BIN} ({expected_size/1e6:.1f} MB, sha256 {output_sha256})")

## Ship it

1. Download `CardsIndexVectors-parity-v2.bin` from the Drive folder.
2. Replace `mobile-apps/ios/TCGer/TCGer/Resources/ScanIndex/CardsIndexVectors.bin` with it (keep the original name).
3. Rebuild and reinstall the iOS app. `CardsIndexMetadata.json`, the gate, and the CoreML model are unchanged.
4. Do **not** replace the web index (`frontend/public/scan-index/pokemon-embeddings.json`) — the browser runs the q8 ONNX model, so its index must stay q8-built.
5. Delete `parity-rebuild-v2-checkpoint.npz` from Drive once you're happy with the result.